In [3]:
# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
        "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261006 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migration-aou-rw-9f879fc4
wb-silky-artichoke-2408.C2024Q3R9



In [1]:
# Load packages
import os
import hail as hl
import pandas as pd 

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
        "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

ModuleNotFoundError: No module named 'hail'

In [2]:
# Run this cell once
hl.init(
    gcs_requester_pays_configuration=os.environ['GOOGLE_CLOUD_PROJECT']
)

NameError: name 'hl' is not defined

In [3]:
# Donwload exclusion samples 
!gcloud storage cp gs://vwb-aou-datasets-controlled/v8/known_issues/wgs_v8_known_issue_1.txt . --billing-project=${GOOGLE_CLOUD_PROJECT}

Copying gs://vwb-aou-datasets-controlled/v8/known_issues/wgs_v8_known_issue_1.txt to file://./wgs_v8_known_issue_1.txt
  Completed files 1/1 | 35.0B/35.0B                                            


In [4]:
# Exclusion samples 
exclusion = pd.read_csv('wgs_v8_known_issue_1.txt', sep='\t')
samples_to_exclude = exclusion['research_id'].values

In [5]:
# ============================================================
# Load MatrixTable
# ============================================================

mt_wgs_path = (
    "gs://vwb-aou-datasets-controlled/"
    "v8/wgs/short_read/snpindel/exome/splitMT/hail.mt"
)

mt = hl.read_matrix_table(mt_wgs_path)


# ============================================================
# Prepare exclusion samples ONCE
# ============================================================

samples_to_exclude = {
    str(x) for x in samples_to_exclude
}

print(f"Samples to exclude: {len(samples_to_exclude)}")


# ============================================================
# Process regions
# ============================================================

variant_ids = hl.literal(set(variant_ids))

mt = mt.filter_rows(
    variant_ids.contains(
        hl.str(mt.locus) + ":" +
        mt.alleles[0] + ":" +
        mt.alleles[1]
    )
)


# --------------------------------------------------------
# Remove excluded samples
# --------------------------------------------------------

mt = mt.filter_cols(
    ~hl.literal(samples_to_exclude).contains(
        mt_region.s
    )
)


# --------------------------------------------------------
# Keep only called ALT genotypes
# --------------------------------------------------------

ht = ht.filter(
    hl.is_defined(ht.GT) &
    (ht.GT.n_alt_alleles() > 0)
)

# --------------------------------------------------------
# Calculate DP
# --------------------------------------------------------

ht = ht.annotate(
    DP=hl.if_else(
        hl.is_defined(ht.AD),
        hl.sum(ht.AD),
        hl.missing(hl.tint32)
    )
)

# --------------------------------------------------------
# Calculate VAF and AB
# --------------------------------------------------------

ht = ht.annotate(
    VAF=hl.if_else(
        hl.is_defined(ht.AD) &
        (ht.DP > 0),
        ht.AD[1] / ht.DP,
        hl.missing(hl.tfloat64)
    ),

    AB=hl.if_else(
        hl.is_defined(ht.AD) &
        (ht.DP > 0) &
        ht.GT.is_het(),
        ht.AD[1] / ht.DP,
        hl.missing(hl.tfloat64)
    )
)

# --------------------------------------------------------
# Remove locus/s columns from table key
#
# entries() creates a key on locus + s.
# This allows us to select locus and s as ordinary fields.
# --------------------------------------------------------



# --------------------------------------------------------
# Increase partitions before export
# --------------------------------------------------------

print("Repartitioning entry table...")

ht = ht.repartition(
    100,
    shuffle=True
)


output = (
    f'{os.environ["WORKSPACE_BUCKET"]}/data/AB_0.35_DP_20/'
    f"variant_genotype_information_"
    f"{region_name}.tsv.bgz"
)

# --------------------------------------------------------
# Export
# --------------------------------------------------------

print(f"Exporting {output}...")

ht.export(output)

print(f"Finished {output}")

Samples to exclude: 3

Processing chr2:25230000-25320000
Repartitioning entry table...
Exporting gs://rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr2_25230000_25320000.tsv.bgz...


Finished gs://rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr2_25230000_25320000.tsv.bgz

Processing chr4:105140000-105280000
Repartitioning entry table...
Exporting gs://rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr4_105140000_105280000.tsv.bgz...


Finished gs://rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr4_105140000_105280000.tsv.bgz

Processing chr5:177510000-177520000
Repartitioning entry table...
Exporting gs://rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr5_177510000_177520000.tsv.bgz...


Finished gs://rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr5_177510000_177520000.tsv.bgz

Processing chr15:90080000-90110000
Repartitioning entry table...
Exporting gs://rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr15_90080000_90110000.tsv.bgz...


Finished gs://rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr15_90080000_90110000.tsv.bgz


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/miniconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/conda/miniconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/conda/miniconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/opt/conda/miniconda3/lib/python3.11/site-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "/opt/conda/miniconda3/lib/python3.11/asyncio/base_events.py", line 608, in run_forever
    self._run_once()
  File "/opt/conda/miniconda3/lib/python3.11/asyncio/base_events.py", line 1921, in _run_once
    handle = self._ready.popleft()
             ^^^^^^^^^^^^^^^^^^^^^
IndexError: pop from an empty deque